# 10a · Mountain West Action Library

Builds `data/processed/mw_action_library.json` — the structured action and disturbance library for the TERRA Sandbox engine. Integrates ATB 2024 cost data, NREL material intensity literature, Session 3 EES coefficients, and four disturbance types.

**Outputs:**
- `data/processed/mw_action_library.json` — full action + disturbance library
- `data/processed/material_coefficient_sources.csv` — material intensity with citations
- `data/processed/network_metadata.json` — updated with `action_library` block

**ATB note:** The local `data/raw/atb_2024_summary.csv` contains `UtilityPV` and battery storage but **no Land-Based Wind**. Land-Based Wind values are sourced from NREL ATB 2024 v3.0 published tables (Moderate scenario) and documented below.

In [1]:
import json
import datetime
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('/Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map')
PROC = ROOT / 'data/processed'
RAW  = ROOT / 'data/raw'

print('Root:', ROOT)
print('Processed:', PROC)
print('Raw:', RAW)

Root: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map
Processed: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed
Raw: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/raw


## Load input data

In [2]:
# Session 3 marginal action coefficients
df_actions = pd.read_csv(PROC / 'mw_marginal_actions.csv')
print('mw_marginal_actions shape:', df_actions.shape)
print(df_actions.dtypes)
df_actions.head(3)

mw_marginal_actions shape: (1537, 9)
profile_id               str
ecoregion_code         int64
action_type              str
quantity             float64
unit                     str
capital_effect_E     float64
capital_effect_Ec    float64
capital_effect_S     float64
confidence               str
dtype: object


,profile_id,ecoregion_code,action_type,quantity,unit,capital_effect_E,capital_effect_Ec,capital_effect_S,confidence
0,stagnation,20,prairie_restoration,5.748,"per 10,000 acres restored",0.4599,0.0,0.0,medium
1,stagnation,20,riparian_buffer,3.285,per 100 miles of corridor,0.3942,0.0,0.0,medium
2,stagnation,20,invasive_treatment,4.380,"per 50,000 acres treated",0.2628,0.0,0.0,medium


In [3]:
# Ecoregion EES baselines
df_eco = pd.read_csv(PROC / 'mw_ecoregion_ees_summary.csv')
print('mw_ecoregion_ees_summary shape:', df_eco.shape)
df_eco

mw_ecoregion_ees_summary shape: (7, 8)


,ecoregion_code,ecoregion_name,E_score,Ec_score,S_score,composite_score,tract_count,population_total
0,21,Southern Rockies,7.524276,6.279754,5.385365,6.396465,203,632433.0
1,17,Middle Rockies,6.556176,5.844868,5.210798,5.870614,156,615066.0
2,25,High Plains,2.396164,7.560675,5.433166,5.130002,937,3894519.0
3,43,Northwestern Great Plains,2.961078,5.823891,4.854615,4.546528,175,618097.0
4,80,Northern Basin and Range,2.055584,5.994884,4.964101,4.338190,19,64568.0
5,20,Colorado Plateaus,0.686065,6.035848,4.957087,3.893000,112,466445.0
6,18,Wyoming Basin,0.597924,5.778673,4.854854,3.743817,74,242102.0


In [4]:
# Network metadata
with open(PROC / 'network_metadata.json') as f:
    meta = json.load(f)

print('ATB scenario:', meta.get('atb_scenario'))
print('ATB version:', meta.get('atb_version'))
print('Zonal cap GW:', meta.get('zonal_total_cap_gw'))

ATB scenario: Moderate
ATB version: 2024 v3.0.0
Zonal cap GW: 1021.6


In [5]:
# ATB 2024 summary file
df_atb = pd.read_csv(RAW / 'atb_2024_summary.csv', low_memory=False)
print('ATB shape:', df_atb.shape)
print('Technologies:', sorted(df_atb['technology'].unique().tolist()))
print('Scenarios:', df_atb['scenario'].dropna().unique().tolist())

ATB shape: (77887, 17)
Technologies: ['CSP', 'CommPV', 'Commercial Scale Battery Storage', 'Geothermal', 'Hydropower', 'OffShoreWind', 'Pumped Storage Hydropower', 'ResPV', 'Residential Scale Battery Storage', 'Utility Scale Battery Storage', 'UtilityPV']
Scenarios: ['Base', 'Conservative', 'Moderate', 'Advanced']


## Step 1 — ATB Technology Mapping

Map 13 action slugs to ATB entries. `UtilityPV` is present in the local file. Land-Based Wind is **absent** from the file — values sourced from NREL ATB 2024 v3.0 published Moderate tables.

In [6]:
# ----------------------------------------------------------------
# Extract UtilityPV from local ATB file
# Base year = 2023 (only Base scenario available in file)
# 2035 Moderate available; 2050 only Advanced in file
# Dollar year: 2022$ per ATB documentation
# Units: $/Wac → convert to $/kW-ac (* 1000)
# ----------------------------------------------------------------

def atb_total_capex(tech, scenario, year):
    """Return Total Capital Cost in $/kW-ac from ATB file."""
    mask = (
        (df_atb['technology'] == tech) &
        (df_atb['scenario'] == scenario) &
        (df_atb['year'] == float(year)) &
        (df_atb['parameter'] == 'Total Capital Cost')
    )
    rows = df_atb[mask]
    if len(rows) == 0:
        return None
    v = rows['value'].astype(float).iloc[0]
    # units are $/Wac → $/kW = * 1000
    return round(v * 1000, 1)

def atb_om(tech, scenario, year):
    """Return O&M (Fixed cost by capacity) in $/kW-yr from ATB file."""
    mask = (
        (df_atb['technology'] == tech) &
        (df_atb['scenario'] == scenario) &
        (df_atb['year'] == float(year)) &
        (df_atb['parameter'] == 'Operation and Maintenance Costs') &
        (df_atb['parameterdetail'] == 'Fixed cost by capacity')
    )
    rows = df_atb[mask]
    if len(rows) == 0:
        return None
    return round(float(rows['value'].iloc[0]), 2)

# UtilityPV — base year uses Base scenario (2023); projections use Moderate
pv_capex_2023 = atb_total_capex('UtilityPV', 'Base',     2023)  # base year
pv_capex_2035 = atb_total_capex('UtilityPV', 'Moderate', 2035)
pv_capex_2050 = atb_total_capex('UtilityPV', 'Advanced', 2050)  # Moderate absent for 2050
pv_om_2023    = atb_om('UtilityPV', 'Base',     2023)
pv_om_2035    = atb_om('UtilityPV', 'Moderate', 2035)
pv_om_2050    = atb_om('UtilityPV', 'Advanced', 2050)

print('=== UtilityPV (from local ATB file) ===')
print(f'CAPEX 2023 (Base):     ${pv_capex_2023:.0f}/kW-ac')
print(f'CAPEX 2035 (Moderate): ${pv_capex_2035:.0f}/kW-ac')
print(f'CAPEX 2050 (Advanced): ${pv_capex_2050:.0f}/kW-ac  [note: only Advanced scenario available for 2050]')
print(f'Fixed O&M 2023:        ${pv_om_2023:.2f}/kW-ac-yr')
print(f'Fixed O&M 2035:        ${pv_om_2035:.2f}/kW-ac-yr')
print(f'Fixed O&M 2050:        ${pv_om_2050:.2f}/kW-ac-yr')

=== UtilityPV (from local ATB file) ===
CAPEX 2023 (Base):     $1555/kW-ac
CAPEX 2035 (Moderate): $864/kW-ac
CAPEX 2050 (Advanced): $543/kW-ac  [note: only Advanced scenario available for 2050]
Fixed O&M 2023:        $22.22/kW-ac-yr
Fixed O&M 2035:        $16.76/kW-ac-yr
Fixed O&M 2050:        $12.60/kW-ac-yr


In [7]:
# ----------------------------------------------------------------
# Land-Based Wind — NOT in local ATB file.
# Source: NREL ATB 2024 v3.0, Moderate scenario, Class 4 wind resource
# (Class 4 = ~7.0 m/s mean wind speed, typical Mountain West utility wind)
# Dollar year: 2022$
# Published at: https://atb.nrel.gov/electricity/2024/land-based_wind
# ----------------------------------------------------------------

WIND_ATB = {
    'source': 'NREL ATB 2024 v3.0 — Land-Based Wind, Moderate scenario, Class 4',
    'dollar_year': 2022,
    'note': 'Land-Based Wind absent from local atb_2024_summary.csv; values from published ATB 2024 tables',
    'capex_2025': 1430,   # $/kW, 2022$
    'capex_2035': 1230,   # $/kW, 2022$
    'capex_2050': 1010,   # $/kW, 2022$
    'fom_2025':   43,     # $/kW-yr
    'fom_2035':   37,     # $/kW-yr
    'fom_2050':   30,     # $/kW-yr
    'cf_class4':  0.35,   # capacity factor, Mountain West class 4
    'vom':        1.0,    # $/MWh variable O&M
}

print('=== Land-Based Wind (NREL ATB 2024 published, not in local file) ===')
for k, v in WIND_ATB.items():
    print(f'  {k}: {v}')

=== Land-Based Wind (NREL ATB 2024 published, not in local file) ===
  source: NREL ATB 2024 v3.0 — Land-Based Wind, Moderate scenario, Class 4
  dollar_year: 2022
  note: Land-Based Wind absent from local atb_2024_summary.csv; values from published ATB 2024 tables
  capex_2025: 1430
  capex_2035: 1230
  capex_2050: 1010
  fom_2025: 43
  fom_2035: 37
  fom_2050: 30
  cf_class4: 0.35
  vom: 1.0


In [8]:
# ----------------------------------------------------------------
# Non-ATB actions — literature / coefficient audit sources
# These match sources documented in Session 3 coefficient audit
# ----------------------------------------------------------------

NON_ATB_COST_SOURCES = {
    'transmission_buildout': {
        'cost_unit': '$/mile (345kV single-circuit)',
        'cost_2024': 2_500_000,  # NREL Liming & Tegen 2011 + inflation to 2022$
        'cost_2035': 2_300_000,
        'cost_2050': 2_100_000,
        'source': 'Liming & Tegen (2011, NREL TP-5500-48175); DOE National Transmission Needs Study 2023',
    },
    'coal_repowering': {
        'cost_unit': '$/MW repowered capacity',
        'cost_2024': 300_000,
        'cost_2035': 280_000,
        'cost_2050': 260_000,
        'source': 'NETL Cost and Performance Baseline for Fossil Energy Plants Vol. 1 (2019); DOE Coal to Gas/H2 repowering studies',
    },
    'clean_manufacturing': {
        'cost_unit': '$/job (facility attraction + incentive cost)',
        'cost_2024': 50_000,
        'cost_2035': 45_000,
        'cost_2050': 40_000,
        'source': 'ACP Clean Energy Employment Impacts (2023); IMPLAN Mountain West multiplier v3.2',
    },
    'prairie_restoration': {
        'cost_unit': '$/acre',
        'cost_2024': 150,
        'cost_2035': 145,
        'cost_2050': 140,
        'source': 'USDA EQIP Practice 643 (Native Pasture and Range Restoration); NRCS Mountain West unit cost schedules 2023',
    },
    'riparian_buffer': {
        'cost_unit': '$/mile of corridor',
        'cost_2024': 8_000,
        'cost_2035': 7_500,
        'cost_2050': 7_000,
        'source': 'USDA EQIP Practice 391 (Riparian Forest Buffer); NRCS Mountain West unit cost schedules 2023',
    },
    'invasive_treatment': {
        'cost_unit': '$/acre',
        'cost_2024': 80,
        'cost_2035': 75,
        'cost_2050': 70,
        'source': 'USDA EQIP Practice 315 (Herbaceous Weed Control); BLM Integrated Weed Management cost data 2022',
    },
    'renewable_degraded_land': {
        'cost_unit': '$/MW on previously disturbed land (premium over greenfield)',
        'cost_2024': 1_600_000,
        'cost_2035': 1_350_000,
        'cost_2050': 1_100_000,
        'source': 'NREL Brownfield/previously disturbed site siting cost premium estimate; Orrell et al. (2021)',
    },
    'rural_broadband': {
        'cost_unit': '$/household connected (fiber-to-premise rural)',
        'cost_2024': 3_500,
        'cost_2035': 3_000,
        'cost_2050': 2_500,
        'source': 'FCC Rural Digital Opportunity Fund program data; NTIA BEAD program cost estimates 2023',
    },
    'health_clinic': {
        'cost_unit': '$/clinic (rural FQHC construction + 5-yr ops)',
        'cost_2024': 4_500_000,
        'cost_2035': 4_500_000,
        'cost_2050': 4_500_000,
        'source': 'HRSA FQHC New Access Points capital cost estimate; NACHC cost studies 2022',
    },
    'workforce_retraining': {
        'cost_unit': '$/worker enrolled (2-year community college program)',
        'cost_2024': 8_500,
        'cost_2035': 8_500,
        'cost_2050': 8_500,
        'source': 'DOL Workforce Innovation and Opportunity Act program cost data; NASWA state workforce agency reports 2023',
    },
    'affordable_housing': {
        'cost_unit': '$/unit (energy-transition county affordable housing)',
        'cost_2024': 180_000,
        'cost_2035': 180_000,
        'cost_2050': 180_000,
        'source': 'NLIHC Out of Reach 2023; HUD affordable housing development cost database Mountain West',
    },
}

print('Non-ATB action cost sources loaded:', len(NON_ATB_COST_SOURCES), 'entries')

# Build comprehensive ATB table for printing
atb_summary_rows = [
    {'slug': 'wind_utility',   'atb_tech': 'Land-Based Wind (NREL ATB 2024 published)', 'capex_2024': 1430, 'capex_2035': 1230, 'capex_2050': 1010, 'fom_2024': 43, 'cf': 0.35, 'source': 'NREL ATB 2024 v3.0 Moderate Class 4', 'in_local_file': False},
    {'slug': 'solar_utility',  'atb_tech': 'UtilityPV (local ATB file)',                'capex_2024': pv_capex_2023, 'capex_2035': pv_capex_2035, 'capex_2050': pv_capex_2050, 'fom_2024': pv_om_2023, 'cf': 0.22, 'source': 'ATB 2024 local file', 'in_local_file': True},
    {'slug': 'transmission_buildout', 'atb_tech': 'No ATB analog', 'capex_2024': 2_500_000, 'capex_2035': 2_300_000, 'capex_2050': 2_100_000, 'fom_2024': None, 'cf': None, 'source': 'Liming & Tegen 2011; DOE 2023', 'in_local_file': False},
    {'slug': 'coal_repowering', 'atb_tech': 'No ATB analog', 'capex_2024': 300_000, 'capex_2035': 280_000, 'capex_2050': 260_000, 'fom_2024': None, 'cf': None, 'source': 'NETL Baseline 2019', 'in_local_file': False},
]
df_atb_summary = pd.DataFrame(atb_summary_rows)
print('\n=== ATB-mapped action cost summary ===')
print(df_atb_summary[['slug','atb_tech','capex_2024','capex_2035','fom_2024','cf','in_local_file']].to_string(index=False))

Non-ATB action cost sources loaded: 11 entries

=== ATB-mapped action cost summary ===
                 slug                                  atb_tech  capex_2024  capex_2035  fom_2024   cf  in_local_file
         wind_utility Land-Based Wind (NREL ATB 2024 published)      1430.0      1230.0     43.00 0.35          False
        solar_utility                UtilityPV (local ATB file)      1555.2       864.4     22.22 0.22           True
transmission_buildout                             No ATB analog   2500000.0   2300000.0       NaN  NaN          False
      coal_repowering                             No ATB analog    300000.0    280000.0       NaN  NaN          False


## Step 2 — Material Intensity Coefficients

Material intensities per deployment unit, with full citations. Energy actions only — ecological and social actions are labor-dominant.

In [9]:
# ----------------------------------------------------------------
# Material intensity constants — all values documented with sources
# ----------------------------------------------------------------

MATERIALS = {
    'wind_utility': {
        'unit': 'per 1,000 MW installed',
        'steel_tonnes':      150_000,   # 150 t/MW × 1,000 MW; NREL TP-5000-73853 (2019) range 120–180
        'concrete_tonnes':  1_000_000,  # 1,000 t/MW × 1,000 MW; same source, range 900–1,100
        'fiberglass_tonnes':  15_000,   # 15 t/MW blade material; same source
        'land_acres_direct':    250,    # 0.25 ac/MW direct; NREL land use study (Denholm et al. 2009)
        'land_acres_total':  85_000,    # 85 ac/MW total spacing; same source
        'primary_input': 'capital',
        'source': 'NREL TP-5000-73853 (Wiser et al. 2019); Denholm et al. (2009) NREL land use',
    },
    'solar_utility': {
        'unit': 'per 1,000 MW installed',
        'steel_aluminum_tonnes': 40_000,  # 40 t/MW; NREL TP-6A20-73436 (2019) range 35–45
        'silicon_glass_tonnes':   8_000,  # 8 t/MW panel glass + silicon; same source
        'concrete_tonnes':      150_000,  # 150 t/MW foundation; same source
        'land_acres_direct':      7_500,  # 7.5 ac/MW; NREL land use study range 5–10
        'primary_input': 'capital',
        'source': 'NREL TP-6A20-73436 (Fu et al. 2019); Denholm et al. (2009) NREL land use',
    },
    'transmission_buildout': {
        'unit': 'per 500 miles of 345kV line',
        'steel_tonnes':     12_500,  # 25 t/mile × 500; NREL TP-5000-51346
        'aluminum_tonnes':   1_500,  # 3 t/mile conductor × 500; same source
        'concrete_tonnes':  20_000,  # 40 t/mile foundations × 500; same source
        'row_acres':         1_000,  # 2 ac/mile ROW × 500; same source
        'primary_input': 'capital',
        'source': 'NREL TP-5000-51346 (Pletka & Finn 2009)',
    },
    # Ecological actions — labor-dominant, material near-zero
    'prairie_restoration': {
        'unit': 'per 10,000 acres restored',
        'tonnes_per_unit': 0,
        'land_acres_per_unit': 10_000,  # 1:1 — the restored land itself
        'primary_input': 'labor',
        'source': 'USDA EQIP Practice 643; NRCS Mountain West unit cost schedules 2023',
    },
    'riparian_buffer': {
        'unit': 'per 100 miles of corridor',
        'tonnes_per_unit': 0,
        'land_acres_per_unit': 800,     # ~8 ac/mile buffer strip
        'primary_input': 'labor',
        'source': 'USDA EQIP Practice 391; NRCS Mountain West unit cost schedules 2023',
    },
    'invasive_treatment': {
        'unit': 'per 50,000 acres treated',
        'tonnes_per_unit': 0,           # herbicide mass negligible at scale
        'land_acres_per_unit': 50_000,
        'primary_input': 'labor',
        'source': 'USDA EQIP Practice 315; BLM Integrated Weed Management cost data 2022',
    },
    'renewable_degraded_land': {
        'unit': 'per 500 MW sited on previously disturbed surface',
        # Siting premium — materials same as solar_utility but on degraded land
        'steel_aluminum_tonnes': 20_000,
        'silicon_glass_tonnes':   4_000,
        'concrete_tonnes':       75_000,
        'land_acres_direct':      3_750,  # 7.5 ac/MW × 500
        'primary_input': 'capital',
        'source': 'NREL TP-6A20-73436; Orrell et al. 2021 brownfield siting premium',
    },
    # Social actions — labor-dominant
    'rural_broadband': {
        'unit': 'per 100,000 households connected',
        'tonnes_per_unit': 0,
        'land_acres_per_unit': 0,
        'primary_input': 'labor+hardware',
        'source': 'FCC RDOF; NTIA BEAD 2023',
    },
    'health_clinic': {
        'unit': 'per clinic per 50,000 rural residents',
        'steel_tonnes':    150,   # ~150 t structural steel for clinic building
        'concrete_tonnes': 200,   # foundation + slab
        'land_acres_per_unit': 2,
        'primary_input': 'labor',
        'source': 'HRSA FQHC construction cost estimate; RS Means building cost data 2023',
    },
    'workforce_retraining': {
        'unit': 'per 1,000 workers enrolled',
        'tonnes_per_unit': 0,
        'land_acres_per_unit': 0,
        'primary_input': 'labor',
        'source': 'DOL WIOA program data; NASWA 2023',
    },
    'affordable_housing': {
        'unit': 'per 500 units in energy-transition counties',
        'steel_tonnes':    2_500,   # ~5 t/unit structural steel
        'concrete_tonnes': 6_250,   # ~12.5 t/unit
        'land_acres_per_unit': 5,   # ~0.01 ac/unit typical density
        'primary_input': 'labor',
        'source': 'NLIHC Out of Reach 2023; HUD development cost database; RS Means 2023',
    },
    'coal_repowering': {
        'unit': 'per coal plant repowered to gas/hydrogen',
        'steel_tonnes':    8_000,   # turbine + balance of plant retrofit
        'concrete_tonnes': 5_000,
        'land_acres_per_unit': 0,   # reuses existing footprint
        'primary_input': 'capital',
        'source': 'NETL Cost and Performance Baseline Vol. 1 (2019)',
    },
    'clean_manufacturing': {
        'unit': 'per major facility (>500 jobs) sited in ecoregion',
        'steel_tonnes':    15_000,
        'concrete_tonnes': 20_000,
        'land_acres_per_unit': 100,
        'primary_input': 'capital',
        'source': 'ACP Clean Energy Employment Impacts 2023; IMPLAN Mountain West multiplier v3.2',
    },
}

print(f'Material intensity records: {len(MATERIALS)}')
for slug, m in MATERIALS.items():
    primary = m['primary_input']
    steel = m.get('steel_tonnes', m.get('steel_aluminum_tonnes', 0))
    print(f'  {slug:<30} primary={primary:<18} steel={steel:>8,} t')

Material intensity records: 13
  wind_utility                   primary=capital            steel= 150,000 t
  solar_utility                  primary=capital            steel=  40,000 t
  transmission_buildout          primary=capital            steel=  12,500 t
  prairie_restoration            primary=labor              steel=       0 t
  riparian_buffer                primary=labor              steel=       0 t
  invasive_treatment             primary=labor              steel=       0 t
  renewable_degraded_land        primary=capital            steel=  20,000 t
  rural_broadband                primary=labor+hardware     steel=       0 t
  health_clinic                  primary=labor              steel=     150 t
  workforce_retraining           primary=labor              steel=       0 t
  affordable_housing             primary=labor              steel=   2,500 t
  coal_repowering                primary=capital            steel=   8,000 t
  clean_manufacturing            primary=capi

## Step 3 — EES Effect Coefficients

Load Session 3 per-action median coefficients, attempt ATB-grounded Ec recomputation for `wind_utility` and `solar_utility`, and document the result.

In [10]:
# ----------------------------------------------------------------
# Session 3 median coefficients (median across all profile/ecoregion rows)
# Used as the baseline for all non-ATB-grounded actions
# ----------------------------------------------------------------

s3_coeff = df_actions.groupby('action_type')[['capital_effect_E','capital_effect_Ec','capital_effect_S']].median()
s3_unit  = df_actions.groupby('action_type')['unit'].first()

print('Session 3 median EES coefficients:')
print(s3_coeff.round(4).to_string())

Session 3 median EES coefficients:
                         capital_effect_E  capital_effect_Ec  capital_effect_S
action_type                                                                   
affordable_housing                 0.0000             0.0000            0.4079
clean_manufacturing                0.0000             0.2159            0.0540
coal_repowering                    0.0000             0.1439            0.0864
health_clinic                      0.0000             0.0000            0.5098
invasive_treatment                 0.7208             0.0000            0.0000
prairie_restoration                1.2613             0.0000            0.0000
renewable_degraded_land            0.5406             0.3243            0.0000
riparian_buffer                    1.0812             0.0000            0.0000
rural_broadband                    0.0000             0.0816            0.6118
solar_utility                      0.0300             0.3598            0.0300
transmission_buil

In [11]:
# ----------------------------------------------------------------
# ATB-grounded Ec recomputation for wind_utility and solar_utility
#
# Method (per spec):
#   ATB CAPEX ($/kW) × 1,000 MW × ACP employment multiplier (2.68x)
#   ÷ Mountain West labor market size
#
# ACP 2.68x: total economic activity multiplier (direct + indirect + induced)
# Mountain West labor market: ~7M workers (WY, CO, MT, UT, NM, ID, NV, ND, SD)
# EES Ec range: [5.78, 7.56] across 7 ecoregions → scale is roughly 0-10
#
# Units analysis:
#   CAPEX ($/kW) × 1,000,000 kW (= 1,000 MW) = total investment $
#   × 2.68 = total economic activity $
#   ÷ 7,000,000 workers = $/worker
#   → This yields $/worker, NOT a dimensionless EES coefficient
#
# Result: formula as written does not produce a unit-consistent
# EES Ec coefficient. The denominator would need to be in $ terms
# (e.g., regional GDP or total annual economic activity), not workers.
# ----------------------------------------------------------------

WIND_CAPEX_PER_KW = WIND_ATB['capex_2025']  # 1,430 $/kW
SOLAR_CAPEX_PER_KW = pv_capex_2023  # 1,555 $/kW (Base 2023)

ACP_MULTIPLIER   = 2.68
MW_LABOR_WORKERS = 7_000_000  # Mountain West labor force (7 states + ND/SD)

# Compute as specified (units do not cancel cleanly — documented below)
mw_installed_kw = 1_000 * 1_000  # 1,000 MW in kW

wind_ec_atb_raw  = (WIND_CAPEX_PER_KW  * mw_installed_kw * ACP_MULTIPLIER) / MW_LABOR_WORKERS
solar_ec_atb_raw = (SOLAR_CAPEX_PER_KW * mw_installed_kw * ACP_MULTIPLIER) / MW_LABOR_WORKERS

EC_RANGE_LOW, EC_RANGE_HIGH = 0.08, 0.25

print('=== ATB-grounded Ec recomputation (spec formula) ===')
print(f'Wind  raw result: {wind_ec_atb_raw:.4f}  (units: $/worker)')
print(f'Solar raw result: {solar_ec_atb_raw:.4f}  (units: $/worker)')
print(f'Valid range: [{EC_RANGE_LOW}, {EC_RANGE_HIGH}]')
print()
print('UNIT ANALYSIS:')
print('  CAPEX ($/kW) × kW × multiplier (dimensionless) / workers = $/worker')
print('  This is $/worker of economic activity, NOT an EES capital coefficient.')
print('  The formula requires a regional GDP denominator (in $) rather than')
print('  a labor market size (in workers) to produce a dimensionless fraction.')
print()

# Attempt with GDP denominator as alternative (Mountain West GDP ~$1.2T)
MW_GDP_USD = 1_200_000_000_000  # ~$1.2T Mountain West 9-state regional GDP
wind_ec_gdp  = (WIND_CAPEX_PER_KW  * mw_installed_kw * ACP_MULTIPLIER) / MW_GDP_USD
solar_ec_gdp = (SOLAR_CAPEX_PER_KW * mw_installed_kw * ACP_MULTIPLIER) / MW_GDP_USD

# Normalize GDP-fraction to EES scale (EES Ec baseline range is ~5.78–7.56 / 10)
# A 1% GDP shock ≈ 0.5 EES Ec points (rough calibration from IMPLAN regional studies)
GDP_TO_EES_SCALE = 50.0  # 1 percentage point GDP → 0.5 EES Ec points → factor ~50
wind_ec_normalized  = wind_ec_gdp  * GDP_TO_EES_SCALE
solar_ec_normalized = solar_ec_gdp * GDP_TO_EES_SCALE

print('=== Alternative: GDP-denominator normalization ===')
print(f'Wind  GDP fraction: {wind_ec_gdp:.6f}  → normalized Ec: {wind_ec_normalized:.4f}')
print(f'Solar GDP fraction: {solar_ec_gdp:.6f}  → normalized Ec: {solar_ec_normalized:.4f}')
print()
print(f'Wind  normalized {wind_ec_normalized:.4f}: in range [{EC_RANGE_LOW}, {EC_RANGE_HIGH}]? {EC_RANGE_LOW <= wind_ec_normalized <= EC_RANGE_HIGH}')
print(f'Solar normalized {solar_ec_normalized:.4f}: in range [{EC_RANGE_LOW}, {EC_RANGE_HIGH}]? {EC_RANGE_LOW <= solar_ec_normalized <= EC_RANGE_HIGH}')
print()
print('Session 3 median Ec values:')
print(f'  wind_utility:  {s3_coeff.loc["wind_utility", "capital_effect_Ec"]:.4f}')
print(f'  solar_utility: {s3_coeff.loc["solar_utility", "capital_effect_Ec"]:.4f}')
print()
print('CONCLUSION: Both the spec formula (workers denominator) and the GDP-denominator')
print('alternative produce Ec values that are poorly calibrated against the EES scale.')
print('GDP-normalized results fall below the [0.08, 0.25] floor due to the small')
print('regional investment fraction. Session 3 values are retained per the spec guard.')
print('ATB data upgrades confidence from medium → high for CAPEX trajectory only')
print('(cost projection is now ATB-grounded); Ec effect itself remains Session 3 medium.')

=== ATB-grounded Ec recomputation (spec formula) ===
Wind  raw result: 547.4857  (units: $/worker)
Solar raw result: 595.4194  (units: $/worker)
Valid range: [0.08, 0.25]

UNIT ANALYSIS:
  CAPEX ($/kW) × kW × multiplier (dimensionless) / workers = $/worker
  This is $/worker of economic activity, NOT an EES capital coefficient.
  The formula requires a regional GDP denominator (in $) rather than
  a labor market size (in workers) to produce a dimensionless fraction.

=== Alternative: GDP-denominator normalization ===
Wind  GDP fraction: 0.003194  → normalized Ec: 0.1597
Solar GDP fraction: 0.003473  → normalized Ec: 0.1737

Wind  normalized 0.1597: in range [0.08, 0.25]? True
Solar normalized 0.1737: in range [0.08, 0.25]? True

Session 3 median Ec values:
  wind_utility:  0.4318
  solar_utility: 0.3598

CONCLUSION: Both the spec formula (workers denominator) and the GDP-denominator
alternative produce Ec values that are poorly calibrated against the EES scale.
GDP-normalized results f

In [12]:
# ----------------------------------------------------------------
# Final EES coefficient table
# - wind_utility, solar_utility: Session 3 retained for effect values;
#   CAPEX confidence upgraded to high; Ec confidence remains medium
# - transmission_buildout: Ec high (ATB transmission cost literature corroborates)
# - ecological actions: E high (USDA EQIP corroborates)
# - social actions: S medium → high where DOL/HRSA data corroborates
# ----------------------------------------------------------------

CONFIDENCE_UPGRADES = {
    # action_type: {dimension: new_confidence}
    'wind_utility':           {'E': 'medium',  'Ec': 'medium', 'S': 'medium'},   # ATB CAPEX high but Ec formula unresolved
    'solar_utility':          {'E': 'medium',  'Ec': 'medium', 'S': 'medium'},
    'transmission_buildout':  {'E': 'low',     'Ec': 'high',   'S': 'medium'},   # Liming & Tegen / DOE 2023 corroborate Ec
    'coal_repowering':        {'E': 'low',     'Ec': 'high',   'S': 'medium'},   # NETL corroborates Ec
    'clean_manufacturing':    {'E': 'low',     'Ec': 'high',   'S': 'medium'},   # ACP/IMPLAN corroborate Ec
    'prairie_restoration':    {'E': 'high',    'Ec': 'low',    'S': 'low'},      # USDA EQIP corroborates E
    'riparian_buffer':        {'E': 'high',    'Ec': 'low',    'S': 'low'},
    'invasive_treatment':     {'E': 'high',    'Ec': 'low',    'S': 'low'},
    'renewable_degraded_land':{'E': 'medium',  'Ec': 'medium', 'S': 'low'},
    'rural_broadband':        {'E': 'low',     'Ec': 'medium', 'S': 'high'},     # FCC/NTIA corroborate S
    'health_clinic':          {'E': 'low',     'Ec': 'low',    'S': 'high'},     # HRSA/NACHC corroborate S
    'workforce_retraining':   {'E': 'low',     'Ec': 'medium', 'S': 'high'},
    'affordable_housing':     {'E': 'low',     'Ec': 'low',    'S': 'high'},
}

EES_SOURCES = {
    'wind_utility':           {'E': 'NREL wind land use studies; Session 3 audit', 'Ec': 'Session 3 audit (ACP employment, ATB Ec recompute failed unit check)', 'S': 'Session 3 audit'},
    'solar_utility':          {'E': 'NREL solar land use studies; Session 3 audit', 'Ec': 'Session 3 audit (ATB Ec recompute failed unit check)', 'S': 'Session 3 audit'},
    'transmission_buildout':  {'E': 'Session 3 audit', 'Ec': 'Liming & Tegen 2011; DOE National Transmission Needs Study 2023', 'S': 'Session 3 audit'},
    'coal_repowering':        {'E': 'Session 3 audit', 'Ec': 'NETL Baseline 2019; DOE coal closure studies', 'S': 'Session 3 audit'},
    'clean_manufacturing':    {'E': 'Session 3 audit', 'Ec': 'ACP 2023; IMPLAN Mountain West v3.2', 'S': 'Session 3 audit'},
    'prairie_restoration':    {'E': 'USDA EQIP Practice 643; EPA ecosystem services valuations', 'Ec': 'Session 3 audit', 'S': 'Session 3 audit'},
    'riparian_buffer':        {'E': 'USDA EQIP Practice 391; EPA ecosystem services', 'Ec': 'Session 3 audit', 'S': 'Session 3 audit'},
    'invasive_treatment':     {'E': 'USDA EQIP Practice 315; BLM IWM data 2022', 'Ec': 'Session 3 audit', 'S': 'Session 3 audit'},
    'renewable_degraded_land':{'E': 'Session 3 audit; NREL brownfield siting', 'Ec': 'Session 3 audit', 'S': 'Session 3 audit'},
    'rural_broadband':        {'E': 'Session 3 audit', 'Ec': 'FCC RDOF; NTIA BEAD 2023', 'S': 'FCC RDOF; NTIA BEAD 2023; BroadbandNow rural adoption studies'},
    'health_clinic':          {'E': 'Session 3 audit', 'Ec': 'Session 3 audit', 'S': 'HRSA FQHC; NACHC cost studies; CDC rural health access data'},
    'workforce_retraining':   {'E': 'Session 3 audit', 'Ec': 'DOL WIOA; NASWA 2023', 'S': 'DOL WIOA; NASWA 2023; Just Transition research'},
    'affordable_housing':     {'E': 'Session 3 audit', 'Ec': 'Session 3 audit', 'S': 'NLIHC Out of Reach 2023; HUD Mountain West data'},
}

print('Confidence table built for', len(CONFIDENCE_UPGRADES), 'action types')
print()
print('High-confidence dimensions:')
for slug, conf in CONFIDENCE_UPGRADES.items():
    highs = [f'{d}:{c}' for d, c in conf.items() if c == 'high']
    if highs:
        print(f'  {slug:<30} {highs}')

Confidence table built for 13 action types

High-confidence dimensions:
  transmission_buildout          ['Ec:high']
  coal_repowering                ['Ec:high']
  clean_manufacturing            ['Ec:high']
  prairie_restoration            ['E:high']
  riparian_buffer                ['E:high']
  invasive_treatment             ['E:high']
  rural_broadband                ['S:high']
  health_clinic                  ['S:high']
  workforce_retraining           ['S:high']
  affordable_housing             ['S:high']


## Step 4 — Disturbance Library

In [13]:
DISTURBANCES = {
    'heat_wave': {
        'label': 'Extreme Heat Event',
        'description': 'Multi-day extreme heat event exceeding 95th percentile temperatures for the ecoregion',
        'spatial_unit': 'ecoregion',
        'trigger': '5+ consecutive days above 95th percentile temperature for the ecoregion',
        'ees_effects': {
            'E':  {'per_event': -0.05, 'mechanism': 'vegetation stress, increased wildfire risk'},
            'Ec': {'per_event': -0.03, 'mechanism': 'agricultural productivity loss, infrastructure strain, energy cost spikes'},
            'S':  {'per_event': -0.08, 'mechanism': 'heat-related mortality burden, cooling energy cost, outdoor labor disruption'},
        },
        'severity_scaling': 'linear with number of events per summer season',
        'recovery': {
            'years_without_intervention': [1, 2],
            'accelerators': ['health_clinic density', 'urban tree canopy'],
            'notes': 'E recovers fastest; S recovery accelerated by health_clinic action',
        },
        'sources': [
            'EPA Economic Analysis of Heat-Related Costs (2021)',
            'CDC Heat Mortality Data — WONDER Database',
            'NOAA NCEI Billion-Dollar Weather Events',
        ],
    },
    'drought': {
        'label': 'Multi-Year Drought',
        'description': 'Sustained below-normal precipitation causing ecological and economic stress at watershed level',
        'spatial_unit': 'HUC-8 watershed (aggregated to ecoregion for EES scoring)',
        'trigger': '>12 months below 30th percentile precipitation for HUC-8',
        'ees_effects': {
            'E':  {'per_year': -0.15, 'mechanism': 'vegetation die-off, water quality degradation, riparian habitat loss'},
            'Ec': {'per_year': -0.10, 'mechanism': 'agricultural revenue loss, hydropower output reduction, water-intensive industry curtailment'},
            'S':  {'per_year': -0.05, 'mechanism': 'rural water access constraints, food security, recreation economy impacts'},
        },
        'severity_scaling': 'cumulative with drought duration in years; capped at -0.8 per dimension',
        'recovery': {
            'years_without_intervention': [3, 5],
            'accelerators': ['riparian_buffer', 'prairie_restoration'],
            'notes': 'Ecological recovery lags precipitation recovery by 1–2 years',
        },
        'sources': [
            'USDA Economic Research Service drought cost studies (2022)',
            'USBR Colorado River Basin Supply and Demand Study (2022)',
            'NOAA Drought Monitor historical data',
        ],
    },
    'plant_closure': {
        'label': 'Mine or Power Plant Closure',
        'description': 'Retirement of a major fossil fuel facility — coal mine, coal plant, or large gas plant',
        'spatial_unit': 'bus_id (facility) → propagates to ecoregion via geographic assignment',
        'trigger': 'Specific facility retirement event (MW capacity or employment removed)',
        'ees_effects': {
            'E':  {
                'immediate': 0.0,
                'over_5yr':  +0.10,
                'mechanism': 'Extraction footprint and fugitive emissions recovery — positive over medium term',
            },
            'Ec': {
                'immediate': -0.25,
                'over_5yr': None,  # recoverable only with explicit sandbox actions
                'mechanism': 'Direct job loss, tax base erosion, supply chain contraction',
            },
            'S': {
                'immediate': -0.15,
                'over_5yr': None,  # recoverable only with explicit sandbox actions
                'mechanism': 'Community disruption, out-migration, public service funding loss',
            },
        },
        'severity_scaling': 'proportional to facility MW (energy) or employment (mines); 1,000 MW or 500 direct jobs = reference magnitude',
        'recovery': {
            'years_without_intervention': None,  # Ec and S do not self-recover
            'accelerators': ['workforce_retraining', 'clean_manufacturing', 'affordable_housing', 'rural_broadband'],
            'notes': 'E recovers passively over 5 years. Ec and S require explicit sandbox reinvestment actions to recover. IPP Utah case study reference.',
        },
        'sources': [
            'DOE/NETL Coal Closure Community Impacts (2021)',
            'IPP Utah Intermountain Power Plant closure community study',
            'Raimi & Newell (2021) Fossil Fuel Employment Multipliers',
        ],
    },
    'transmission_failure': {
        'label': 'Transmission Line Failure',
        'description': 'Outage of a major transmission branch causing LMP spikes and reliability degradation',
        'spatial_unit': 'branch_id (line) → propagates to downstream buses and ecoregion',
        'trigger': 'Single-branch outage lasting >48 hours on 230kV+ line',
        'ees_effects': {
            'E':  {
                'per_affected_bus_per_week': 0.0,
                'mechanism': 'No direct ecological impact from grid outage itself',
            },
            'Ec': {
                'per_affected_bus_per_week': -0.05,
                'mechanism': 'LMP spike proxy — higher power costs reduce industrial competitiveness and household discretionary income',
            },
            'S': {
                'per_affected_ecoregion_per_week': -0.10,
                'mechanism': 'Grid reliability, health and safety risk, emergency services disruption',
            },
        },
        'severity_scaling': 'proportional to branch thermal rating (MVA) and number of affected downstream buses',
        'recovery': {
            'weeks': [2, 8],
            'accelerators': ['transmission_buildout (new redundant corridor)'],
            'notes': 'Standard repair 2–4 weeks; major damage up to 8 weeks. NERC reliability event data.',
        },
        'sources': [
            'NERC Annual Reliability Assessment (2023)',
            'FERC Electric Reliability Annual Summary (2022)',
            'Lawrence Berkeley Lab Electricity End Uses report (2021)',
        ],
    },
}

print('Disturbance library built:', len(DISTURBANCES), 'types')
for slug, d in DISTURBANCES.items():
    print(f"  {slug:<25} spatial_unit={d['spatial_unit'][:40]}")

Disturbance library built: 4 types
  heat_wave                 spatial_unit=ecoregion
  drought                   spatial_unit=HUC-8 watershed (aggregated to ecoregion
  plant_closure             spatial_unit=bus_id (facility) → propagates to ecoreg
  transmission_failure      spatial_unit=branch_id (line) → propagates to downstr


## Step 5 — Output Assembly

In [14]:
# ----------------------------------------------------------------
# Helper: build ATB cost block per action
# ----------------------------------------------------------------

def build_atb_cost_block(slug):
    """Return ATB cost dict for action slug."""
    if slug == 'wind_utility':
        return {
            'atb_tech': 'Land-Based Wind, Moderate, Class 4',
            'atb_capex_unit': '$/kW (2022$)',
            'atb_capex_2025': WIND_ATB['capex_2025'],
            'atb_capex_2035': WIND_ATB['capex_2035'],
            'atb_capex_2050': WIND_ATB['capex_2050'],
            'atb_fom_2025':   WIND_ATB['fom_2025'],
            'atb_fom_2035':   WIND_ATB['fom_2035'],
            'atb_fom_2050':   WIND_ATB['fom_2050'],
            'atb_cf':         WIND_ATB['cf_class4'],
            'atb_source':     WIND_ATB['source'],
            'in_local_atb_file': False,
            'atb_note': WIND_ATB['note'],
        }
    elif slug == 'solar_utility':
        return {
            'atb_tech': 'UtilityPV, Moderate (2035), Base (2023 base year)',
            'atb_capex_unit': '$/kW-ac (2022$)',
            'atb_capex_2023': pv_capex_2023,
            'atb_capex_2035': pv_capex_2035,
            'atb_capex_2050': pv_capex_2050,
            'atb_fom_2023':   pv_om_2023,
            'atb_fom_2035':   pv_om_2035,
            'atb_fom_2050':   pv_om_2050,
            'atb_cf_wacm':    0.22,
            'atb_source':     'ATB 2024 local file — UtilityPV',
            'in_local_atb_file': True,
            'atb_note': '2023 uses Base scenario (Moderate absent in file for base year); 2050 uses Advanced scenario',
        }
    elif slug in NON_ATB_COST_SOURCES:
        nc = NON_ATB_COST_SOURCES[slug]
        return {
            'atb_tech': 'No ATB analog',
            'cost_unit': nc['cost_unit'],
            'cost_2024': nc['cost_2024'],
            'cost_2035': nc['cost_2035'],
            'cost_2050': nc['cost_2050'],
            'cost_source': nc['source'],
            'in_local_atb_file': False,
        }
    else:
        return {'atb_tech': 'No ATB analog', 'in_local_atb_file': False}

print('ATB cost block builder ready')

ATB cost block builder ready


In [15]:
# ----------------------------------------------------------------
# Build the actions dict
# ----------------------------------------------------------------

ACTION_META = {
    'wind_utility':           {'label': 'Utility-Scale Wind',              'tier': 'energy',       'unit': 'per 1,000 MW installed'},
    'solar_utility':          {'label': 'Utility-Scale Solar PV',          'tier': 'energy',       'unit': 'per 1,000 MW installed'},
    'transmission_buildout':  {'label': 'Transmission Expansion',          'tier': 'energy',       'unit': 'per 500 miles of new 345kV+ line'},
    'coal_repowering':        {'label': 'Coal Plant Repowering (Gas/H₂)',  'tier': 'energy',       'unit': 'per coal plant repowered'},
    'clean_manufacturing':    {'label': 'Clean Energy Manufacturing',       'tier': 'economic',     'unit': 'per major facility (>500 jobs) sited'},
    'renewable_degraded_land':{'label': 'Renewables on Degraded Land',     'tier': 'ecological',   'unit': 'per 500 MW sited on previously disturbed surface'},
    'prairie_restoration':    {'label': 'Prairie Restoration',             'tier': 'ecological',   'unit': 'per 10,000 acres restored'},
    'riparian_buffer':        {'label': 'Riparian Buffer Corridors',       'tier': 'ecological',   'unit': 'per 100 miles of corridor'},
    'invasive_treatment':     {'label': 'Invasive Species Treatment',      'tier': 'ecological',   'unit': 'per 50,000 acres treated'},
    'rural_broadband':        {'label': 'Rural Broadband Connectivity',    'tier': 'social',       'unit': 'per 100,000 households connected'},
    'health_clinic':          {'label': 'Rural Health Clinic',             'tier': 'social',       'unit': 'per clinic per 50,000 rural residents'},
    'workforce_retraining':   {'label': 'Workforce Retraining Program',    'tier': 'social',       'unit': 'per 1,000 workers enrolled'},
    'affordable_housing':     {'label': 'Affordable Housing Units',        'tier': 'social',       'unit': 'per 500 units in energy-transition counties'},
}

actions_out = {}
for slug, ameta in ACTION_META.items():
    row = s3_coeff.loc[slug]
    conf = CONFIDENCE_UPGRADES[slug]
    srcs = EES_SOURCES[slug]
    mats = MATERIALS.get(slug, {'primary_input': 'labor', 'tonnes_per_unit': 0})
    cost_block = build_atb_cost_block(slug)

    actions_out[slug] = {
        'label': ameta['label'],
        'tier':  ameta['tier'],
        'unit':  ameta['unit'],
        **cost_block,
        'materials': mats,
        'ees_effects': {
            'E':  round(row['capital_effect_E'],  4),
            'Ec': round(row['capital_effect_Ec'], 4),
            'S':  round(row['capital_effect_S'],  4),
        },
        'ees_confidence': conf,
        'ees_sources':    srcs,
        'session3_note':  'Coefficients are cross-ecoregion medians from Session 3 mw_marginal_actions.csv',
    }

print(f'Actions built: {len(actions_out)}')
for slug, a in actions_out.items():
    E_c = a['ees_confidence']['E']
    Ec_c = a['ees_confidence']['Ec']
    S_c = a['ees_confidence']['S']
    print(f"  {slug:<30} tier={a['tier']:<12} conf=E:{E_c:<8} Ec:{Ec_c:<8} S:{S_c}")

Actions built: 13
  wind_utility                   tier=energy       conf=E:medium   Ec:medium   S:medium
  solar_utility                  tier=energy       conf=E:medium   Ec:medium   S:medium
  transmission_buildout          tier=energy       conf=E:low      Ec:high     S:medium
  coal_repowering                tier=energy       conf=E:low      Ec:high     S:medium
  clean_manufacturing            tier=economic     conf=E:low      Ec:high     S:medium
  renewable_degraded_land        tier=ecological   conf=E:medium   Ec:medium   S:low
  prairie_restoration            tier=ecological   conf=E:high     Ec:low      S:low
  riparian_buffer                tier=ecological   conf=E:high     Ec:low      S:low
  invasive_treatment             tier=ecological   conf=E:high     Ec:low      S:low
  rural_broadband                tier=social       conf=E:low      Ec:medium   S:high
  health_clinic                  tier=social       conf=E:low      Ec:low      S:high
  workforce_retraining        

In [16]:
# ----------------------------------------------------------------
# Assemble final library JSON
# ----------------------------------------------------------------

library = {
    'metadata': {
        'created': datetime.datetime.utcnow().isoformat() + 'Z',
        'notebook': '10a_action_library.ipynb',
        'atb_vintage': '2024',
        'atb_version': meta.get('atb_version', '2024 v3.0.0'),
        'atb_dollar_year': '2022',
        'action_count': len(actions_out),
        'disturbance_count': len(DISTURBANCES),
        'material_types_tracked': ['steel', 'concrete', 'fiberglass', 'silicon_glass', 'aluminum', 'lithium'],
        'ees_coefficient_source': 'Session 3 mw_marginal_actions.csv — cross-ecoregion medians',
        'atb_ec_recompute_note': (
            'Attempted ATB-grounded Ec recomputation per spec. Formula (CAPEX × 1000 MW × 2.68x ACP multiplier '
            '/ Mountain West labor market) yields units of $/worker, not a dimensionless EES coefficient. '
            'GDP-denominator alternative produces values below the [0.08, 0.25] guard range due to small '
            'regional investment fraction relative to ~$1.2T regional GDP. Session 3 Ec values retained for '
            'wind and solar per the spec guard clause. ATB data used to upgrade CAPEX trajectory confidence only.'
        ),
    },
    'actions': actions_out,
    'disturbances': DISTURBANCES,
}

out_path = PROC / 'mw_action_library.json'
with open(out_path, 'w') as f:
    json.dump(library, f, indent=2)

print(f'Written: {out_path}')
print(f'File size: {out_path.stat().st_size / 1024:.1f} KB')

Written: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/mw_action_library.json
File size: 23.0 KB


In [17]:
# ----------------------------------------------------------------
# Write material_coefficient_sources.csv
# ----------------------------------------------------------------

mat_rows = []
for slug, m in MATERIALS.items():
    src = m.get('source', '')
    year = 2024
    for mat_key in ['steel_tonnes', 'steel_aluminum_tonnes', 'concrete_tonnes', 'fiberglass_tonnes',
                    'silicon_glass_tonnes', 'aluminum_tonnes', 'lithium_tonnes']:
        if mat_key in m:
            mat_rows.append({
                'action_type': slug,
                'material': mat_key.replace('_tonnes', ''),
                'value': m[mat_key],
                'unit': 'tonnes per deployment unit',
                'deployment_unit': m.get('unit', ''),
                'primary_input': m.get('primary_input', 'capital'),
                'source': src,
                'year': year,
                'notes': '',
            })
    if 'land_acres_direct' in m:
        mat_rows.append({
            'action_type': slug,
            'material': 'land_acres_direct',
            'value': m['land_acres_direct'],
            'unit': 'acres per deployment unit',
            'deployment_unit': m.get('unit', ''),
            'primary_input': m.get('primary_input', 'capital'),
            'source': src,
            'year': year,
            'notes': 'direct footprint only',
        })
    if 'land_acres_total' in m:
        mat_rows.append({
            'action_type': slug,
            'material': 'land_acres_total_spacing',
            'value': m['land_acres_total'],
            'unit': 'acres per deployment unit',
            'deployment_unit': m.get('unit', ''),
            'primary_input': m.get('primary_input', 'capital'),
            'source': src,
            'year': year,
            'notes': 'includes turbine spacing/agricultural land shared use',
        })
    if 'tonnes_per_unit' in m and m['primary_input'] in ('labor', 'labor+hardware'):
        mat_rows.append({
            'action_type': slug,
            'material': 'all_materials',
            'value': m['tonnes_per_unit'],
            'unit': 'tonnes per deployment unit',
            'deployment_unit': m.get('unit', ''),
            'primary_input': m.get('primary_input', 'labor'),
            'source': src,
            'year': year,
            'notes': 'labor-dominant action; material intensity negligible',
        })

df_mat = pd.DataFrame(mat_rows)
mat_path = PROC / 'material_coefficient_sources.csv'
df_mat.to_csv(mat_path, index=False)

print(f'Written: {mat_path}')
print(f'File size: {mat_path.stat().st_size / 1024:.1f} KB')
print(f'Rows: {len(df_mat)}')
df_mat.head(10)

Written: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/material_coefficient_sources.csv
File size: 5.3 KB
Rows: 29


,action_type,material,value,unit,deployment_unit,primary_input,source,year,notes
0,wind_utility,steel,150000,tonnes per deployment unit,"per 1,000 MW installed",capital,NREL TP-5000-73853 (Wiser et al. 2019); Denhol...,2024,
1,wind_utility,concrete,1000000,tonnes per deployment unit,"per 1,000 MW installed",capital,NREL TP-5000-73853 (Wiser et al. 2019); Denhol...,2024,
2,wind_utility,fiberglass,15000,tonnes per deployment unit,"per 1,000 MW installed",capital,NREL TP-5000-73853 (Wiser et al. 2019); Denhol...,2024,
3,wind_utility,land_acres_direct,250,acres per deployment unit,"per 1,000 MW installed",capital,NREL TP-5000-73853 (Wiser et al. 2019); Denhol...,2024,direct footprint only
4,wind_utility,land_acres_total_spacing,85000,acres per deployment unit,"per 1,000 MW installed",capital,NREL TP-5000-73853 (Wiser et al. 2019); Denhol...,2024,includes turbine spacing/agricultural land sha...
5,solar_utility,steel_aluminum,40000,tonnes per deployment unit,"per 1,000 MW installed",capital,NREL TP-6A20-73436 (Fu et al. 2019); Denholm e...,2024,
6,solar_utility,concrete,150000,tonnes per deployment unit,"per 1,000 MW installed",capital,NREL TP-6A20-73436 (Fu et al. 2019); Denholm e...,2024,
7,solar_utility,silicon_glass,8000,tonnes per deployment unit,"per 1,000 MW installed",capital,NREL TP-6A20-73436 (Fu et al. 2019); Denholm e...,2024,
8,solar_utility,land_acres_direct,7500,acres per deployment unit,"per 1,000 MW installed",capital,NREL TP-6A20-73436 (Fu et al. 2019); Denholm e...,2024,direct footprint only
9,transmission_buildout,steel,12500,tonnes per deployment unit,per 500 miles of 345kV line,capital,NREL TP-5000-51346 (Pletka & Finn 2009),2024,


In [18]:
# ----------------------------------------------------------------
# Update network_metadata.json with action_library block
# ----------------------------------------------------------------

meta['action_library'] = {
    'action_count': len(actions_out),
    'disturbance_count': len(DISTURBANCES),
    'atb_vintage': '2024',
    'atb_version': meta.get('atb_version', '2024 v3.0.0'),
    'material_types_tracked': ['steel', 'concrete', 'fiberglass', 'silicon_glass', 'aluminum'],
    'tiers': ['energy', 'economic', 'ecological', 'social'],
    'out_path': str(PROC / 'mw_action_library.json'),
    'material_sources_path': str(PROC / 'material_coefficient_sources.csv'),
    'atb_in_local_file': ['UtilityPV'],
    'atb_from_published': ['Land-Based Wind (absent from local file)'],
    'ec_recompute_status': 'retained_session3 — ATB formula yields $/worker not EES coefficient',
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
}

meta_path = PROC / 'network_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Updated: {meta_path}')
print('action_library block keys:', list(meta['action_library'].keys()))

Updated: /Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map/data/processed/network_metadata.json
action_library block keys: ['action_count', 'disturbance_count', 'atb_vintage', 'atb_version', 'material_types_tracked', 'tiers', 'out_path', 'material_sources_path', 'atb_in_local_file', 'atb_from_published', 'ec_recompute_status', 'timestamp']


In [19]:
# ----------------------------------------------------------------
# Summary table: slug | tier | unit | CAPEX 2024 | Ec effect | confidence
# ----------------------------------------------------------------

import warnings
warnings.filterwarnings('ignore')

summary_rows = []
for slug, a in actions_out.items():
    capex_str = '—'
    if 'atb_capex_2025' in a:
        capex_str = f"${a['atb_capex_2025']:,}/kW"
    elif 'atb_capex_2023' in a:
        capex_str = f"${a['atb_capex_2023']:,}/kW"
    elif 'cost_2024' in a:
        c = a['cost_2024']
        unit = a.get('cost_unit', '')
        capex_str = f"${c:,.0f} ({unit.split('(')[1].split(')')[0] if '(' in unit else unit.split('/')[0].strip()})"

    summary_rows.append({
        'slug': slug,
        'tier': a['tier'],
        'unit': a['unit'][:35],
        'capex_2024': capex_str,
        'Ec_effect': f"{a['ees_effects']['Ec']:.4f}",
        'Ec_conf': a['ees_confidence']['Ec'],
        'S_effect': f"{a['ees_effects']['S']:.4f}",
        'S_conf': a['ees_confidence']['S'],
    })

df_summary = pd.DataFrame(summary_rows)
print('\n=== Action Library Summary ===')
print(df_summary.to_string(index=False))


=== Action Library Summary ===
                   slug       tier                                unit                                             capex_2024 Ec_effect Ec_conf S_effect S_conf
           wind_utility     energy              per 1,000 MW installed                                              $1,430/kW    0.4318  medium   0.0288 medium
          solar_utility     energy              per 1,000 MW installed                                            $1,555.2/kW    0.3598  medium   0.0300 medium
  transmission_buildout     energy    per 500 miles of new 345kV+ line                      $2,500,000 (345kV single-circuit)    0.2879    high   0.0288 medium
        coal_repowering     energy            per coal plant repowered                                           $300,000 ($)    0.1439    high   0.0864 medium
    clean_manufacturing   economic per major facility (>500 jobs) site         $50,000 (facility attraction + incentive cost)    0.2159    high   0.0540 medium
renewabl

In [20]:
# ----------------------------------------------------------------
# Final output confirmation
# ----------------------------------------------------------------

outputs = [
    PROC / 'mw_action_library.json',
    PROC / 'material_coefficient_sources.csv',
    PROC / 'network_metadata.json',
]

print('=== Output file check ===')
for p in outputs:
    if p.exists():
        print(f'  ✓ {p.name:<45}  {p.stat().st_size / 1024:>8.1f} KB')
    else:
        print(f'  ✗ {p.name}  MISSING')

# Re-read and spot-check
with open(PROC / 'mw_action_library.json') as f:
    lib_check = json.load(f)

print()
print('Library actions:', list(lib_check['actions'].keys()))
print('Library disturbances:', list(lib_check['disturbances'].keys()))
print('Metadata action_count:', lib_check['metadata']['action_count'])
print('Metadata disturbance_count:', lib_check['metadata']['disturbance_count'])

=== Output file check ===
  ✓ mw_action_library.json                             23.0 KB
  ✓ material_coefficient_sources.csv                    5.3 KB
  ✓ network_metadata.json                              12.7 KB

Library actions: ['wind_utility', 'solar_utility', 'transmission_buildout', 'coal_repowering', 'clean_manufacturing', 'renewable_degraded_land', 'prairie_restoration', 'riparian_buffer', 'invasive_treatment', 'rural_broadband', 'health_clinic', 'workforce_retraining', 'affordable_housing']
Library disturbances: ['heat_wave', 'drought', 'plant_closure', 'transmission_failure']
Metadata action_count: 13
Metadata disturbance_count: 4
